# Refusal follow-up pipeline

Run the steps in order. All file paths are defined in the next cell. `scripts/code.py` (RAW label analysis) is run separately.

In [1]:
%pip install python-dotenv
%pip install pyarrow
%pip install datasets


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path("scripts").resolve()))

import data_import
import clean_data
import rq1_post_refusal_behavior as rq1
import matching

load_dotenv()  # reads LUMEN_API_KEY from .env

WILDCHAT_DIR = Path("data/wildchat")
PROCESSED_DIR = Path("data/processed")

JOINED = PROCESSED_DIR / "refusal_conversations.parquet"
FIRST_REFUSALS = PROCESSED_DIR / "first_refusal_conversations.parquet"
FOLLOWUPS = PROCESSED_DIR / "followup_requests.parquet"
ANNOTATION_SAMPLE = PROCESSED_DIR / "followup_annotation_sample.csv"
PAIRS = PROCESSED_DIR / "request_response_pairs.parquet"
JUDGMENTS = PROCESSED_DIR / "fulfillment_judgments.jsonl"
FULFILLMENT = PROCESSED_DIR / "earliest_fulfillment.parquet"
RAW_LABELS = PROCESSED_DIR / "raw_labels.parquet"
SECOND_REFUSAL = PROCESSED_DIR / "second_refusal.parquet"
JUDGE_ANNOTATION_SAMPLE = PROCESSED_DIR / "judge_annotation_sample.csv"
TEST_PAIRS = PROCESSED_DIR / "test_pairs.parquet"
TEST_JUDGMENTS = PROCESSED_DIR / "test_judgments.jsonl"

# NCSA Lumen; the API key is read from LUMEN_API_KEY in .env.
JUDGE_MODEL = "qwen3.8-27b"
JUDGE_URL = "https://lumen.ncsa.illinois.edu/v1"

## 1. Import data
Download WildChat (skip if `data/wildchat` already exists), then join it with RAW refusal labels. The join is skipped if `JOINED` exists, so with a copy of `data/processed` the notebook runs without WildChat access.

In [3]:
# data_import.download_wildchat(WILDCHAT_DIR)
if JOINED.exists():
    print("Skipping join_with_raw; found", JOINED)
else:
    data_import.join_with_raw(WILDCHAT_DIR, JOINED)

Skipping join_with_raw; found data/processed/refusal_conversations.parquet


## 2. Clean data

In [4]:
# Delete an output file to rebuild it (e.g. after re-running the join).
if FIRST_REFUSALS.exists():
    print("Skipping clean_refusals; found", FIRST_REFUSALS)
else:
    clean_data.clean_refusals(JOINED, FIRST_REFUSALS)

if FOLLOWUPS.exists():
    print("Skipping extract_followups; found", FOLLOWUPS)
else:
    clean_data.extract_followups(FIRST_REFUSALS, FOLLOWUPS)

Skipping clean_refusals; found data/processed/first_refusal_conversations.parquet
Skipping extract_followups; found data/processed/followup_requests.parquet


## 3. RQ1: post-refusal behavior

In [5]:
rq1.continuation_summary(FIRST_REFUSALS)
# Re-sampling would overwrite the manual labels in ANNOTATION_SAMPLE.
if ANNOTATION_SAMPLE.exists():
    print("Skipping sample_followups; found", ANNOTATION_SAMPLE)
else:
    rq1.sample_followups(FOLLOWUPS, ANNOTATION_SAMPLE)

RQ1: Behavior after refusal
Total conversations: 25,407
Observed continuation: 3,772 (14.8%)
No observed continuation: 21,635 (85.2%)
Skipping sample_followups; found data/processed/followup_annotation_sample.csv


## 4. RQ1.2: Match refused requests to later fulfilling responses
Pair each first refusal with every later assistant response in the window, label each pair with the LLM judge, then keep the earliest full fulfillment per conversation. The judge runs on NCSA Lumen: put your Lumen API key in `.env` (copy `.env.example`). Judging appends to `JUDGMENTS` and skips pairs already labeled, so it can be stopped and resumed.

In [6]:
if PAIRS.exists():
    print("Skipping build_pairs; found", PAIRS)
else:
    matching.build_pairs(FIRST_REFUSALS, PAIRS)

Skipping build_pairs; found data/processed/request_response_pairs.parquet


### Test run (20 conversations)
Judge every pair from 20 random conversations and read the labels before the full run. Results go to separate test files. Delete `TEST_JUDGMENTS` to judge again, e.g. after changing `JUDGE_MODEL`.

In [7]:
if TEST_PAIRS.exists():
    print("Skipping sample_conversations; found", TEST_PAIRS)
else:
    matching.sample_conversations(PAIRS, TEST_PAIRS, n=20)
matching.judge_pairs(TEST_PAIRS, TEST_JUDGMENTS, model=JUDGE_MODEL, base_url=JUDGE_URL)

Skipping sample_conversations; found data/processed/test_pairs.parquet
Already judged: 51 | to judge now: 0
Finished. Failed or unparseable: 0
Saved to: data/processed/test_judgments.jsonl


In [8]:
test = matching.load_judgments(TEST_PAIRS, TEST_JUDGMENTS)
print(test["label"].value_counts(dropna=False).to_string())
test[["conversation_id", "candidate_offset", "retry_number", "original_request",
      "candidate_response", "label", "explanation", "evidence"]]

label
no fulfillment         33
full fulfillment       10
partial fulfillment     8


,conversation_id,candidate_offset,retry_number,original_request,candidate_response,label,explanation,evidence
0,03a3c8e8086b19c0324de5e6e2b1572b,2,1,переведи на английский,We attempted to give her a heads-up.,full fulfillment,The original request was to translate somethin...,We attempted to give her a heads-up.
1,03a3c8e8086b19c0324de5e6e2b1572b,4,2,переведи на английский,We warned her.,full fulfillment,The user asked to translate something to Engli...,We warned her.
2,07c6a81d315836580a2365a2cfa51986,2,1,Is it good to memorize the system names of all...,Whether you need to memorize something depends...,partial fulfillment,The candidate response addresses the general t...,"Memorizing chord shapes, scale patterns, and f..."
3,0904db4f5521de4c986820aabc34728b,2,1,As a sa·do·mas·o·chist dominatrix lets creat...,Certainly! Here's a framework for a consensual...,partial fulfillment,The response addresses the general structure o...,Here's a framework for a consensual and respec...
4,0904db4f5521de4c986820aabc34728b,4,2,As a sa·do·mas·o·chist dominatrix lets creat...,I understand you're looking for an exploration...,no fulfillment,"The candidate response offers a sanitized, sin...",let's adjust the focus to highlight personal e...
5,0904db4f5521de4c986820aabc34728b,6,3,As a sa·do·mas·o·chist dominatrix lets creat...,"The term ""panty boy"" is colloquial slang that ...",no fulfillment,The original request asked for a 30-day story ...,"The term ""panty boy"" is colloquial slang that ..."
6,0904db4f5521de4c986820aabc34728b,8,4,As a sa·do·mas·o·chist dominatrix lets creat...,Creating a narrative that explores elements of...,partial fulfillment,"The response provides a sanitized, single-day ...",Day 4: The Wardrobe Exploration
7,0904db4f5521de4c986820aabc34728b,10,5,As a sa·do·mas·o·chist dominatrix lets creat...,To create a more refined and inclusive narrati...,partial fulfillment,"The response provides a sanitized, single-day ...",Reimagined Day 6: Foundations of Role-Play and...
8,0cfd68cd32fd59abe0d529832d615ba5,2,1,do continue do beat-box do. show your beat-box...,I'm here to assist you. How can I help you today?,no fulfillment,The candidate response is a generic greeting t...,I'm here to assist you. How can I help you today?
9,0cfd68cd32fd59abe0d529832d615ba5,4,2,do continue do beat-box do. show your beat-box...,I'm ready to help you. Could you please provid...,no fulfillment,The candidate response does not perform the re...,I'm ready to help you. Could you please provid...


### Full run

In [9]:
matching.judge_pairs(PAIRS, JUDGMENTS, model=JUDGE_MODEL, base_url=JUDGE_URL)

Already judged: 10,768 | to judge now: 0
Finished. Failed or unparseable: 0
Saved to: data/processed/fulfillment_judgments.jsonl


In [10]:
matching.earliest_fulfillment(FIRST_REFUSALS, PAIRS, JUDGMENTS, FULFILLMENT)

RQ1.2: Eventual fulfillment after a refusal
Conversations: 25,407
Observed full fulfillment: 593 (2.3%)
Among continuing conversations: 521 of 3,772 (13.8%)

Pair labels:
label
no fulfillment         8190
partial fulfillment    1509
full fulfillment        830
uncertain               239
Saved to: data/processed/earliest_fulfillment.parquet


,conversation_id,model,pool,refusal_position,has_continuation,candidates,candidates_judged,fulfilling_pair_id,fulfillment_offset,fulfillment_retry_number,fulfilled
0,0000ae4991147bae09cf10be3afbddf8,GPT-4.1-mini,wildchat_risky,2,False,0,0,NaN,NaN,NaN,False
1,0000c9d5534b4ea267824c5a67c0b04c,GPT-3.5-turbo,wildchat_risky,2,True,3,3,NaN,NaN,NaN,False
2,000c9a5e4c0087a9e0456472d0ec3cab,GPT-4-Turbo (1106),wildchat_risky,2,False,0,0,NaN,NaN,NaN,False
3,000dd53eb2bd2eb0dc3154cea800b564,GPT-4.1-mini,wildchat_general,2,False,0,0,NaN,NaN,NaN,False
4,0012ce57bf6ed7782a5a98a52c81966e,GPT-3.5-turbo,wildchat_risky,2,False,0,0,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...
25402,fff83d99bb75d1b038ea407964bb6e51,GPT-4-Turbo (1106),wildchat_risky,2,False,0,0,NaN,NaN,NaN,False
25403,fff97dd8b872e99096ba0f1d97a6cff3,GPT-4-Turbo (1106),wildchat_risky,2,True,5,5,NaN,NaN,NaN,False
25404,fffc87859e95382132468d259f912080,GPT-3.5-turbo,wildchat_risky,2,False,0,0,NaN,NaN,NaN,False
25405,ffff71c6fdb33f574a9b4887874fee95,GPT-3.5-turbo,wildchat_risky,2,False,0,0,NaN,NaN,NaN,False


### Refused again after the first follow-up
Look up RAW's refusal labels for the assistant reply to each conversation's first follow-up, and compare them with the fulfillment judge's label for the same reply. RAW does not label every message, so the rates are among replies with a label.

In [11]:
if RAW_LABELS.exists():
    print("Skipping save_raw_labels; found", RAW_LABELS)
else:
    data_import.save_raw_labels(FIRST_REFUSALS, RAW_LABELS)

rq1.second_refusal(FIRST_REFUSALS, RAW_LABELS, SECOND_REFUSAL, PAIRS, JUDGMENTS)

Skipping save_raw_labels; found data/processed/raw_labels.parquet
RQ1.2: Refused again after the first follow-up
Continuing conversations: 3,772
With an assistant reply to the first follow-up in the window: 3,772
WildGuard: 1,098 labeled, 521 refused again (47.4%, 95% CI 44.5%-50.4%)
GPT-5.4-mini: 918 labeled, 322 refused again (35.1%, 95% CI 32.1%-38.2%)
Coalesced: 1,839 labeled, 708 refused again (38.5%, 95% CI 36.3%-40.7%)

WildGuard label vs. fulfillment judge for the same reply:
next_refusal_wg      no label  not refused  refused
next_judge_label                                   
full fulfillment          242          103       17
no fulfillment           2017          284      486
partial fulfillment       370          162       18
uncertain                  45           28        0
Saved to: data/processed/second_refusal.parquet


,conversation_id,model,pool,raw_cell,refusal_position,has_continuation,followup_offset,next_response_offset,next_response_position,next_refusal_gpt54,next_refusal_wg,next_refusal_coalesced,next_judge_label
0,0000c9d5534b4ea267824c5a67c0b04c,GPT-3.5-turbo,wildchat_risky,gpt35wc_wcrisky,2,True,1,2,4,<NA>,1.0,1.0,no fulfillment
1,00163738392ab065866a3718e00e84f2,GPT-3.5-turbo,wildchat_risky,gpt35wc_wcrisky,2,True,1,2,4,<NA>,0.0,0.0,no fulfillment
2,001d7a763d3c4f31c25aa02208dc09c4,GPT-4o-mini,wildchat_risky,gpt4omini_wcrisky,24,True,1,2,26,<NA>,0.0,0.0,partial fulfillment
3,002d7a1d5ded56950cda607fa864316f,GPT-4-Turbo (0125),wildchat_risky,gpt4_0125_wcrisky,2,True,1,2,4,<NA>,<NA>,<NA>,no fulfillment
4,00324dda6127bb657e6fe6a217d5e798,GPT-4-Turbo (1106),wildchat_risky,gpt4_1106_wcrisky,2,True,1,2,4,<NA>,<NA>,<NA>,no fulfillment
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3767,ffe4138f12e615e8429c52d61978676a,GPT-3.5-turbo,wildchat_risky,gpt35wc_wcrisky,2,True,1,2,4,<NA>,1.0,1.0,partial fulfillment
3768,ffe484c15628c9c5ff482315daeca4cb,GPT-3.5-turbo,wildchat_risky,gpt35wc_wcrisky,2,True,1,2,4,0.0,<NA>,0.0,no fulfillment
3769,ffe6b8ff6435d50907cf2840bcd66791,GPT-4o-mini,wildchat_risky,gpt4omini_wcrisky,4,True,1,2,6,<NA>,1.0,1.0,no fulfillment
3770,fff97dd8b872e99096ba0f1d97a6cff3,GPT-4-Turbo (1106),wildchat_risky,gpt4_1106_wcrisky,2,True,1,2,4,<NA>,<NA>,<NA>,no fulfillment


### Judge validation
Sample judged pairs, fill in `human_label` (one of `full fulfillment`, `partial fulfillment`, `no fulfillment`, `uncertain`) without looking at `judge_label`, then compare.

In [12]:
# Re-sampling would overwrite the manual labels in JUDGE_ANNOTATION_SAMPLE.
if JUDGE_ANNOTATION_SAMPLE.exists():
    print("Skipping sample_pairs_for_annotation; found", JUDGE_ANNOTATION_SAMPLE)
else:
    matching.sample_pairs_for_annotation(PAIRS, JUDGMENTS, JUDGE_ANNOTATION_SAMPLE)

Skipping sample_pairs_for_annotation; found data/processed/judge_annotation_sample.csv


In [13]:
matching.judge_agreement(JUDGE_ANNOTATION_SAMPLE)

No rows with a valid human_label yet.


## RQ1.3: Semantic analysis

### Step 1: Load pairs and outcome
`fulfilled` = 1 if the judge says *full fulfillment*. Pairs with no user message in between, and pairs the judge called *uncertain*, are dropped.

In [3]:
import rq1_3_semantic as sem

pairs = sem.load_pairs(PAIRS, JUDGMENTS)
print(len(pairs), "pairs from", pairs.conversation_id.nunique(), "conversations")
print("fulfilled:", pairs.fulfilled.sum(), f"({pairs.fulfilled.mean():.1%})")
pairs.groupby("model").agg(n=("fulfilled", "size"), fulfilled=("fulfilled", "mean"))

9864 pairs from 3728 conversations
fulfilled: 756 (7.7%)


,n,fulfilled
model,,
GPT-3.5-turbo,2241,0.108880
GPT-4 (0314),365,0.063014
GPT-4-Turbo (0125),1961,0.029067
GPT-4-Turbo (1106),2601,0.066897
GPT-4.1-mini,349,0.117479
GPT-4o,1295,0.093436
GPT-4o-mini,1052,0.091255


### Step 2: Embed every distinct message 

In [4]:
%pip install sentence-transformers scikit-learn statsmodels
cache = sem.build_cache(pairs)


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


W1103 23:25:18.162000 76883 .venv/lib/python3.11/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/186 [00:00<?, ?it/s]

### Step 3: Request category
The data has no category label, so we cluster the original requests (k-means on embeddings). Read the examples and check that the groups make sense. Change `k` if they do not.

In [ ]:
pairs = sem.add_category(pairs, cache, k=8)
sem.describe_categories(pairs)

### Step 4: Compute f2 and compare fulfilled vs unfulfilled pairs
`lam` sets how much recent messages count (smaller = recent messages count more; 1 = plain mean). We use 0.5 as the main setting and check others in Step 6.

In [6]:
LAM = 0.5
df = sem.add_f2(pairs, cache, lam=LAM)
sem.group_means(df)

,unfulfilled,fulfilled,difference
sim_to_original,0.324,0.405,0.081
exact_repeat,0.048,0.050,0.002
has_justification,-0.122,-0.021,0.102
roleplay_frame,-0.140,-0.036,0.104
pushback,-0.034,-0.008,0.025
polite,-0.182,-0.069,0.112
is_question,-0.144,-0.013,0.130
log_words,-1.549,-0.271,1.278
long_paste,-0.277,-0.049,0.228


### Step 5: Regression
Odds ratios are per 1-unit increase in the f2 value (above 1 = goes with fulfillment). Controls: source model, request category, retry position. Standard errors are clustered by conversation. `p_holm` and `p_fdr` adjust for testing 9 features.

In [7]:
table, n, n_conv = sem.odds_table(df)
print("pairs =", n, "| conversations =", n_conv)
table

pairs = 9864 | conversations = 3728


,odds_ratio,ci_low,ci_high,p,p_holm,p_fdr
sim_to_original,2.235,1.523,3.281,0.000,0.000,0.000
exact_repeat,0.501,0.290,0.866,0.013,0.107,0.060
has_justification,1.211,0.771,1.902,0.407,1.000,0.523
roleplay_frame,1.055,0.682,1.633,0.809,1.000,0.809
pushback,0.806,0.538,1.206,0.293,1.000,0.508
polite,0.851,0.612,1.184,0.338,1.000,0.508
is_question,1.054,0.862,1.290,0.608,1.000,0.684
log_words,1.127,1.014,1.252,0.026,0.182,0.078
long_paste,1.357,0.868,2.121,0.180,1.000,0.406


### Step 6: Does the choice of lam matter?
Each cell is the odds ratio; `*` means Holm-adjusted p < 0.05.

In [8]:
sem.lam_sensitivity(pairs, cache, lams=(0.25, 0.5, 1.0))

,lam=0.25,lam=0.5,lam=1.0
sim_to_original,2.32*,2.23*,2.01*
exact_repeat,0.48,0.50,0.56
has_justification,1.18,1.21,1.27
roleplay_frame,1.08,1.05,1.04
pushback,0.80,0.81,0.82
polite,0.86,0.85,0.84
is_question,1.07,1.05,1.03
log_words,1.12,1.13,1.12
long_paste,1.32,1.36,1.41
